In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
uploaded = files.upload()

In [ ]:
uploaded = files.upload()

In [ ]:
# STEP 0

import warnings
warnings.filterwarnings("ignore")        # suppress noisy convergence warnings

import os
import joblib                            # to save/load trained model objects
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Sklearn: splitting & evaluation ---
from sklearn.model_selection import (
    StratifiedKFold,          # 5-fold CV that keeps class ratios balanced
    cross_validate,           # runs CV and returns multiple score metrics
    GridSearchCV,             # exhaustive hyperparameter search
)

# --- Sklearn: preprocessing ---
from sklearn.preprocessing import (
    LabelEncoder,             # converts class names  →  integers
    StandardScaler,           # rescales features to mean=0, std=1
    OrdinalEncoder,           # encodes text categories  →  numeric codes
)
from sklearn.impute import SimpleImputer  # fills missing values

# --- Sklearn: pipeline & column routing ---
from sklearn.pipeline  import Pipeline          # chains steps sequentially
from sklearn.compose   import ColumnTransformer # routes columns to different steps

# --- Sklearn: feature selection ---
from sklearn.feature_selection import (
    VarianceThreshold,        # drops near-constant / zero-variance features
    SelectKBest,              # keeps top-k features by a scoring function
    f_classif,                # ANOVA F-score: measures class separability per feature
)

# --- Sklearn: classifiers ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble     import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,  # XGBoost alternative — built into sklearn,
                                     # zero extra install needed, handles missing
                                     # values natively, same boosting power
)
from sklearn.svm import SVC

# --- Sklearn: metrics (for training-set report only) ---
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
)

In [ ]:
# =============================================================================
# STEP 1 — TRAIN DATA LOADING + VARIABLE NAMES
# =============================================================================

df_train = pd.read_csv(
    "/content/processed_train_vowel_and_cough_ros.csv"
)

print(f"Rows    : {df_train.shape[0]}")
print(f"Columns : {df_train.shape[1]}")

print("\nClass distribution:")
print(df_train["disease"].value_counts().to_string())

TARGET = "disease"

# IMPORTANT:
# id should NOT be used as feature

X_train = df_train.drop(columns=[TARGET, "id"]).copy()
y_train = df_train[TARGET].copy()

print(f"\nFeature matrix shape : {X_train.shape}")
print(f"Target shape         : {y_train.shape}")

In [ ]:
# =============================================================================
# STEP 2 — BUILD THE FEATURE SELECTION PIPELINE
# =============================================================================
# Even after preprocessing, we have 72 features — many of which are correlated
# MFCC variants.  Keeping all of them risks:
#   • overfitting (model memorises noise instead of signal)
#   • slower training
#   • reduced interpretability
#
# We apply two-stage trimming:
#
#   Stage 1 — VarianceThreshold(threshold=0.01)
#     Drops any feature whose variance is less than 0.01 after scaling.
#     After StandardScaler, variance=1 for all features normally, but if a
#     feature was near-constant BEFORE scaling, scaling amplifies noise —
#     so we catch them at this threshold.
#
#   Stage 2 — SelectKBest(f_classif, k=70)
#     Scores every remaining feature with the ANOVA F-test:
#     "How much does this feature's value differ across the 4 disease classes?"
#     Higher F-score = more discriminative = more useful for classification.
#     We keep the top 40 (out of ~72), discarding the weakest 32.
#     k=40 is a starting point — GridSearchCV will also tune this value.

print("=" * 65)
print("STEP 8 : Building the feature selection pipeline")
print("=" * 65)

feature_selection = Pipeline(steps=[
    (
        "var_filter",
        VarianceThreshold(threshold=0.01),
        # Drops near-constant features.  After StandardScaler these are
        # features that were essentially identical across all samples.
    ),
    (
        "kbest",
        SelectKBest(score_func=f_classif, k=80),
        # ANOVA F-test: tests whether a feature's mean differs significantly
        # across disease classes.  Keeps the 40 most class-separating features.
    ),
])

print("  Stage 1 : VarianceThreshold(0.01)  → drops near-constant features")
print("  Stage 2 : SelectKBest(f_classif, k=70) → keeps top 70 by ANOVA F-score")
print()

In [ ]:
# =============================================================================
# STEP 3 — BUILD_FULL_PIPELINE
# =============================================================================
# REPLACE ENTIRE build_full_pipeline() FUNCTION WITH THIS
# =============================================================================

def build_full_pipeline(clf):

    return Pipeline(steps=[

        # ONLY feature selection now
        ("featsel", feature_selection),

        # classifier
        ("clf", clf),

    ])

In [ ]:
# STEP 4

print("=" * 65)
print("STEP 10 : Defining candidate models")
print("=" * 65)

models = {}

# --- Model 1: Logistic Regression ---
models["LogisticRegression"] = build_full_pipeline(
    LogisticRegression(
        C=1.0,                      # inverse regularisation strength; C=1 = mild L2 penalty
        max_iter=2000,              # enough iterations to converge on 40 features
        multi_class="multinomial",  # native multi-class (4 diseases) via softmax
        class_weight="balanced",    # auto-upweights minority classes
        solver="lbfgs",             # efficient solver for multinomial + L2
        random_state=42,
    )
)
print("  [1] LogisticRegression — linear baseline with L2 regularisation")

# --- Model 2: Random Forest ---
models["RandomForest"] = build_full_pipeline(
    RandomForestClassifier(
        n_estimators=200,       # 200 trees — good accuracy/speed tradeoff
        max_depth=None,         # trees grow fully; forest's bagging prevents overfit
        min_samples_leaf=2,     # each leaf needs ≥2 samples — slight smoothing
        class_weight="balanced",
        n_jobs=-1,              # use all CPU cores
        random_state=42,
    )
)
print("  [2] RandomForest — 200 trees, bagging, feature importance available")

# --- Model 3: SVM with RBF kernel ---
models["SVM_RBF"] = build_full_pipeline(
    SVC(
        kernel="rbf",           # radial basis function — non-linear boundary
        C=10,                   # soft margin; allows some misclassifications
        gamma="scale",          # gamma = 1/(n_features * X.var()) — auto-scaled
        probability=True,       # enables .predict_proba() at inference time
        class_weight="balanced",
        random_state=42,
    )
)
print("  [3] SVM (RBF) — non-linear kernel, good for high-dim audio features")

# --- Model 4: Gradient Boosting ---
models["GradientBoosting"] = build_full_pipeline(
    GradientBoostingClassifier(
        n_estimators=200,       # 200 sequential trees
        learning_rate=0.1,      # step size per tree (shrinkage)
        max_depth=5,            # max depth per tree — controls complexity
        subsample=0.8,          # use 80% of samples per tree — reduces variance
        random_state=42,
    )
)
print("  [4] GradientBoosting — sequential tree boosting, sklearn native")

# --- Model 5: HistGradientBoosting (XGBoost alternative, built into sklearn) ---
# WHY HistGradientBoosting:
#   • Zero extra install — already inside sklearn
#   • Uses histogram-based splitting (same idea as LightGBM / XGBoost)
#     which makes it much faster than standard GradientBoostingClassifier
#   • Handles missing values (NaN) natively — no imputer required for it
#   • l2_regularization acts like XGBoost's reg_lambda — penalises large
#     leaf weights and trims the model to prevent overfitting
#   • class_weight="balanced" handles residual imbalance after ROS
#   • On benchmarks it scores within 1-2% of XGBoost on tabular datasets
models["HistGradientBoosting"] = build_full_pipeline(
    HistGradientBoostingClassifier(
        max_iter=300,              # number of boosting rounds (= n_estimators)
        learning_rate=0.05,        # contribution of each tree — smaller = more precise
        max_depth=6,               # max depth of each tree — controls complexity
        min_samples_leaf=20,       # min samples per leaf — smooths the model
        l2_regularization=1.0,     # L2 penalty on leaf weights (trims the model)
        class_weight="balanced",   # auto-upweights minority classes
        random_state=42,
    )
)
print("  [5] HistGradientBoosting — sklearn's built-in XGBoost alternative")

# --- Model 6: XGBoost ---
# WHY XGBoost:
# • Industry-standard gradient boosting — often outperforms sklearn's
#   HistGradientBoosting on tabular data with proper tuning
# • use_label_encoder=False + eval_metric suppress deprecation warnings
# • scale_pos_weight is not used here since class_weight="balanced"
#   equivalent is handled via sample_weight in fit() — instead we pass
#   equal weight and let the pipeline handle it
# • tree_method="hist" mirrors HistGradientBoosting's histogram splitting:
#   faster training, lower memory, same boosting power
# • reg_alpha (L1) + reg_lambda (L2) together give elastic-net-style
#   regularisation — more flexible than sklearn's L2-only option
models["XGBoost"] = build_full_pipeline(
    XGBClassifier(
        n_estimators=300,          # boosting rounds — matches HistGB
        learning_rate=0.05,        # step size per tree (shrinkage)
        max_depth=6,               # tree depth — controls complexity
        subsample=0.8,             # row sampling per tree — reduces variance
        colsample_bytree=0.8,      # feature sampling per tree — reduces correlation
        reg_alpha=0.1,             # L1 regularisation — induces sparsity
        reg_lambda=1.0,            # L2 regularisation — penalises large weights
        use_label_encoder=False,   # suppress deprecation warning
        eval_metric="mlogloss",    # multi-class log loss — appropriate for 4 classes
        tree_method="hist",        # histogram-based splitting: fast + memory-efficient
        random_state=42,
        n_jobs=-1,
    )
)
print(" [6] XGBoost — histogram boosting with L1+L2 regularisation")

print(f"\n  Total models to train: {len(models)}")
print()

In [ ]:
# =============================================================================
# STEP 5 — CROSS-VALIDATION ON TRAINING DATA
# =============================================================================
# We evaluate every model using 5-fold Stratified Cross-Validation on the
# TRAINING set only (you've already held out your test set externally).
#
# Why cross-validate instead of just fitting once?
#   A single fit on all training data gives no indication of how the model
#   generalises. CV gives an unbiased estimate by rotating the held-out fold.
#
# Metrics used:
#   accuracy   — overall % of correct predictions
#   f1_macro   — average F1 across all 4 classes equally weighted;
#                better than accuracy for slightly imbalanced data

print("=" * 65)
print("STEP 11 : Cross-validation on training data  (5-fold stratified)")
print("=" * 65)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_summary = {}

for name, pipe in models.items():
    print(f"\n  → Training [{name}] ...")
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        n_jobs=-1,
        return_train_score=False,   # we only care about validation fold scores
    )
    acc_mean = scores["test_accuracy"].mean()
    acc_std  = scores["test_accuracy"].std()
    f1_mean  = scores["test_f1_macro"].mean()
    f1_std   = scores["test_f1_macro"].std()

    cv_summary[name] = {
        "Accuracy Mean": round(acc_mean, 4),
        "Accuracy Std" : round(acc_std,  4),
        "F1 Macro Mean": round(f1_mean,  4),
        "F1 Macro Std" : round(f1_std,   4),
    }
    print(f"     Accuracy  : {acc_mean:.4f}  ± {acc_std:.4f}")
    print(f"     F1 Macro  : {f1_mean:.4f}  ± {f1_std:.4f}")

# Print summary table
summary_df = (
    pd.DataFrame(cv_summary).T
    .sort_values("F1 Macro Mean", ascending=False)
)
print("\n\n  ── Cross-Validation Summary (sorted by F1 Macro) ──")
print(summary_df.to_string())

# Identify best model by F1 macro
best_model_name = summary_df.index[0]
best_f1         = summary_df.loc[best_model_name, "F1 Macro Mean"]
print(f"\n  ✓ Best model : {best_model_name}  (CV F1 Macro = {best_f1:.4f})")
print()

scores = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=-1,
    return_train_score=False,
)

In [ ]:
# =============================================================================
# STEP 5 — CROSS-VALIDATION ON TRAINING DATA (HistGradientBoosting only)
# =============================================================================

from sklearn.model_selection import StratifiedKFold, cross_validate

print("=" * 65)
print("STEP 11 : HistGradientBoosting CV (5-fold stratified)")
print("=" * 65)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# assuming your HistGradientBoosting pipeline/model is stored in:
# models["HistGradientBoosting"]

hgb_pipe = models["HistGradientBoosting"]

print("\n  → Training [HistGradientBoosting] ...")

scores = cross_validate(
    hgb_pipe,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    n_jobs=-1,
    return_train_score=False,
)

acc_mean = scores["test_accuracy"].mean()
acc_std  = scores["test_accuracy"].std()

f1_mean  = scores["test_f1_macro"].mean()
f1_std   = scores["test_f1_macro"].std()

print(f"     Accuracy  : {acc_mean:.4f} ± {acc_std:.4f}")
print(f"     F1 Macro  : {f1_mean:.4f} ± {f1_std:.4f}")

# optional: fold-wise scores
print("\nFold-wise Accuracy :", scores["test_accuracy"])
print("Fold-wise F1 Macro :", scores["test_f1_macro"])

In [ ]:
# =============================================================================
# STEP 9 — LOAD VALIDATION DATA CORRECTLY
# =============================================================================

df_val = pd.read_csv(
    "/content/processed_val_vowel_and_cough.csv"
)

X_val = df_val.drop(columns=["disease", "id"]).copy()
y_val = df_val["disease"].copy()

print(f"\nValidation shape : {X_val.shape}")

print("\nValidation class distribution:")
print(y_val.value_counts().to_string())

In [ ]:
# =============================================================================
# TRAIN vs VALIDATION LOSS CURVE (HistGradientBoosting ONLY)
# =============================================================================

if best_model_name == "HistGradientBoosting":

    print("\n")
    print("=" * 70)
    print("TRAIN vs VALIDATION LOSS CURVE")
    print("=" * 70)

    # ---------------------------------------------------------
    # Build fresh pipeline
    # ---------------------------------------------------------

    hist_pipe = models["HistGradientBoosting"]

    # ---------------------------------------------------------
    # Feature selection
    # ---------------------------------------------------------

    featsel = hist_pipe.named_steps["featsel"]

    X_train_fs = featsel.fit_transform(X_train, y_train)
    X_val_fs   = featsel.transform(X_val)

    # ---------------------------------------------------------
    # Extract classifier
    # ---------------------------------------------------------

    hist_model = hist_pipe.named_steps["clf"]

    # ---------------------------------------------------------
    # IMPORTANT:
    # enable early stopping + validation tracking
    # ---------------------------------------------------------

    hist_model.set_params(
        early_stopping=True,
        validation_fraction=None,   # because we provide explicit validation
        scoring="loss",
    )

    # ---------------------------------------------------------
    # Train model
    # ---------------------------------------------------------

    hist_model.fit(
        X_train_fs,
        y_train,
    )

    # ---------------------------------------------------------
    # TRAIN LOSS
    # train_score_ stores NEGATIVE loss
    # ---------------------------------------------------------

    train_loss = -hist_model.train_score_

    # ---------------------------------------------------------
    # VALIDATION LOSS
    #
    # HistGradientBoosting does not directly store external
    # validation loss history.
    #
    # So we compute staged validation loss manually.
    # ---------------------------------------------------------

    val_loss = []

    for y_proba in hist_model.staged_predict_proba(X_val_fs):

        loss = log_loss(
            y_val,
            y_proba,
            labels=np.unique(y_train)
        )

        val_loss.append(loss)

    # ---------------------------------------------------------
    # Epochs / iterations
    # ---------------------------------------------------------

    epochs = range(1, len(train_loss) + 1)

    # Match lengths safely
    min_len = min(len(train_loss), len(val_loss))

    train_loss = train_loss[:min_len]
    val_loss   = val_loss[:min_len]
    epochs     = range(1, min_len + 1)

    # ---------------------------------------------------------
    # Plot
    # ---------------------------------------------------------

    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs,
        train_loss,
        label="Training Loss",
        linewidth=2,
    )

    plt.plot(
        epochs,
        val_loss,
        label="Validation Loss",
        linewidth=2,
    )

    plt.xlabel("Boosting Iteration")
    plt.ylabel("Log Loss")
    plt.title("Training vs Validation Loss — HistGradientBoosting")

    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# STEP 7 — DISPLAY CLASS LABEL MAPPING
# =============================================================================
# PLACE THIS BEFORE evaluate_dataset() FUNCTION
# =============================================================================

label_map = {
    "asthma": 0,
    "copd": 1,
    "covid": 2,
    "healthy": 3
}

inverse_label_map = {v: k for k, v in label_map.items()}

print("\n")
print("=" * 70)
print("CLASS LABEL ENCODING")
print("=" * 70)

for disease, number in label_map.items():
    print(f"{disease:<10} → {number}")

print()

# =============================================================================
# STEP 8 — GENERIC EVALUATION FUNCTION
# =============================================================================
# This evaluates:
#
# 1. 4-class classification
#    healthy / asthma / copd / covid
#
# 2. Binary classification
#    healthy vs diseased
#
# It prints:
#   • classification report
#   • macro avg
#   • weighted avg
#   • confusion matrix
#   • accuracy
#
# =============================================================================

def evaluate_dataset(model, X, y, dataset_name):

    print("\n")
    print("=" * 80)
    print(f"{dataset_name.upper()} EVALUATION")
    print("=" * 80)

    # =========================================================================
    # PREDICTIONS
    # =========================================================================

    y_pred = model.predict(X)

    # =========================================================================
    # 4-CLASS EVALUATION
    # =========================================================================

    print("\n")
    print("#" * 70)
    print("VERSION 1 : 4-CLASS CLASSIFICATION")
    print("#" * 70)

    class_names = ["healthy", "asthma", "copd", "covid"]

    print("\nClassification Report:\n")

    print(classification_report(
        y,
        y_pred,
        target_names=class_names,
        digits=4,
    ))

    print("\nExplanation:")
    print("Macro Average    → unweighted mean across all classes")
    print("Weighted Average → weighted by number of samples per class")

    accuracy_4 = accuracy_score(y, y_pred)

    print(f"\nOverall Accuracy : {accuracy_4:.4f}")

    # =========================================================================
    # CONFUSION MATRIX — 4 CLASS
    # =========================================================================

    print("\nShowing 4-class confusion matrices...\n")

    cm = confusion_matrix(y, y_pred)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # raw counts
    ConfusionMatrixDisplay(
        cm,
        display_labels=class_names
    ).plot(ax=axes[0], cmap="Blues", colorbar=True)

    axes[0].set_title(
        f"{dataset_name} — 4-Class Confusion Matrix\n(Counts)"
    )

    # normalized
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    ConfusionMatrixDisplay(
        cm_norm,
        display_labels=class_names
    ).plot(ax=axes[1], cmap="Blues", colorbar=True)

    axes[1].set_title(
        f"{dataset_name} — 4-Class Confusion Matrix\n(Row-Normalized)"
    )

    plt.tight_layout()
    plt.show()

    # =========================================================================
    # BINARY CLASSIFICATION
    # =========================================================================
    # healthy = 0
    # diseased = 1
    #
    # asthma/copd/covid ALL become diseased
    # =========================================================================

    print("\n")
    print("#" * 70)
    print("VERSION 2 : BINARY CLASSIFICATION")
    print("HEALTHY vs DISEASED")
    print("#" * 70)

    # healthy = 3
    # diseased = everything else

    y_binary = (y != 3).astype(int)
    y_pred_binary = (y_pred != 3).astype(int)

    binary_names = ["healthy", "diseased"]

    print("\nClassification Report:\n")

    print(classification_report(
        y_binary,
        y_pred_binary,
        target_names=binary_names,
        digits=4,
    ))

    print("\nExplanation:")
    print("Macro Average    → simple mean across healthy + diseased")
    print("Weighted Average → weighted according to class frequency")

    accuracy_binary = accuracy_score(y_binary, y_pred_binary)

    print(f"\nOverall Accuracy : {accuracy_binary:.4f}")

    # =========================================================================
    # CONFUSION MATRIX — BINARY
    # =========================================================================

    print("\nShowing binary confusion matrices...\n")

    cm_bin = confusion_matrix(y_binary, y_pred_binary)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # raw counts
    ConfusionMatrixDisplay(
        cm_bin,
        display_labels=binary_names
    ).plot(ax=axes[0], cmap="Greens", colorbar=True)

    axes[0].set_title(
        f"{dataset_name} — Binary Confusion Matrix\n(Counts)"
    )

    # normalized
    cm_bin_norm = (
        cm_bin.astype(float)
        / cm_bin.sum(axis=1, keepdims=True)
    )

    ConfusionMatrixDisplay(
        cm_bin_norm,
        display_labels=binary_names
    ).plot(ax=axes[1], cmap="Greens", colorbar=True)

    axes[1].set_title(
        f"{dataset_name} — Binary Confusion Matrix\n(Row-Normalized)"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# VALIDATION EVALUATION
# =============================================================================

evaluate_dataset(
    model=models[best_model_name],
    X=X_val,
    y=y_val,
    dataset_name="Validation"
)

In [ ]:
# =============================================================================
# STEP 10 — LOAD TEST DATA
# =============================================================================

df_test = pd.read_csv(
    "processed_test_vowel_and_cough.csv"
)

X_test = df_test.drop(columns=["disease", "id"]).copy()
y_test = df_test["disease"].copy()

print(f"\nTest shape : {X_test.shape}")

print("\nTest class distribution:")
print(y_test.value_counts().to_string())

# =============================================================================
# TEST EVALUATION
# =============================================================================

evaluate_dataset(
    model=models[best_model_name],
    X=X_test,
    y=y_test,
    dataset_name="Test"
)